# Current Project Audit

Repository inventory, data lineage, implementation matrix, blocking coverage, candidate diagnostics, duplicate-candidate checks, censoring diagnostics, and manual-validation readiness.

This notebook is a front-end for the reproducible script `scripts/audit_current_project.py`. The script remains the canonical execution unit; this notebook is for inspection, reruns, and discussion.

In [1]:
from pathlib import Path
import subprocess
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
TABLES_DIR = PROJECT_ROOT / 'reports' / 'tables'
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
print(PROJECT_ROOT)

/home/senghakrou/survival-analysis


## Run the analysis

Set `RUN_SCRIPT = True` when you want to regenerate the outputs. Leave it as `False` for quick review of existing tables.

In [2]:
RUN_SCRIPT = False

if RUN_SCRIPT:
    result = subprocess.run(['python3', 'scripts/audit_current_project.py'], cwd=PROJECT_ROOT, text=True, capture_output=True, check=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)

## Key outputs

### `audit_credibility_implementation_matrix.csv`

One row per credibility check, with implementation status and priority.

In [3]:
path = TABLES_DIR / 'audit_credibility_implementation_matrix.csv'
df = pd.read_csv(path)
print(df.shape)
display(df.head(20))

(26, 8)


,check_name,scientific_question,status,evidence_in_code,evidence_in_outputs,main_assumptions,problems_found,priority
0,Eligibility definition,Which source contracts can enter linkage?,IMPLEMENTED AND VERIFIED,preprocess_boamp_m0.py,boamp_m0_sources.csv,APPEL_OFFRE digital scope; nonmissing buyer ke...,duration imputation affects estimated end dates,P1
1,Blocking coverage,How many eligible sources can be compared?,PARTIALLY IMPLEMENTED,build_m0_candidate_pairs.py,audit_blocking_coverage.csv,same buyer_key and duration-centered temporal ...,zero-candidate causes were absent before audit,P1
2,Candidate-pool diagnostics,Are candidate pools ambiguous?,PARTIALLY IMPLEMENTED,candidate rank/margin columns,audit_candidate_source_diagnostics.csv,top two scores summarize ambiguity,no source-level table before audit,P1
3,Duplicate-candidate analysis,Is one candidate reused by many sources?,PARTIALLY IMPLEMENTED,run_m0_linkage.py duplicate count,audit_duplicate_candidate_*.csv,many-to-one may be valid for lots/frameworks,examples/distribution absent before audit,P1
4,Score and margin diagnostics,Which components drive decisions?,PARTIALLY IMPLEMENTED,m0_score_distribution_summary.csv,audit_score_component_distributions.csv,internal scores are not validation,accepted/rejected/runner-up populations absent...,P1
5,Threshold sensitivity,Do cutoffs alter events?,IMPLEMENTED BUT INCONSISTENT,eval_m6a_threshold_sensitivity.py,m6a_threshold_*.csv,percentile thresholds,hardcoded eligible denominator and limited bre...,P1
6,Temporal-window sensitivity,Does the window drive links?,NOT IMPLEMENTED,none found,pending eval_m6d_temporal_window_sensitivity.py,rerun candidate generation for each W,single 6-month window only,P1
7,Feature ablation,Which components change rankings?,PARTIALLY IMPLEMENTED,eval_m6b_feature_ablation.py,m6b_feature_ablation.csv,"drop component, renormalize, rerank",missing selected-candidate/event-status/surviv...,P2
8,Weight sensitivity,Do manual weights matter?,NOT IMPLEMENTED,none found,none,weights are manual,no perturbation grid,P2
9,Fellegi-Sunter mixture,Model-based precision conditional on Omega?,IMPLEMENTED AND VERIFIED,eval_m3_fellegi_sunter_mixture.py,m3_*.csv,unsupervised beta mixture,not verified accuracy; beta imperfect for disc...,P2


### `audit_dataset_inventory.csv`

Official datasets, row/column counts, current/stale status, and checksums.

In [4]:
path = TABLES_DIR / 'audit_dataset_inventory.csv'
df = pd.read_csv(path)
print(df.shape)
display(df.head(20))

(10, 9)


,dataset,path,description,unit_of_observation,creation_script,row_count,column_count,current_or_stale,sha256
0,raw_pdl_json,data/raw/boamp/pdl,monthly raw JSON,monthly export,download_boamp.py,140,NaN,current,NaN
1,raw_national_archive,data/raw/boamp_national_2024_2026_archive,superseded national raw JSON,monthly export,none active,32,NaN,stale/superseded,NaN
2,interim_flattened,data/interim/boamp_raw_flattened.csv,flattened notices,notice,parse_boamp.py,84623,42.0,current,451931d739e16fc5b7d5a1664ea8a803de9e1a48ea89e1...
3,cleaned_notices,data/processed/boamp_clean_m0_no_enrichment.csv,cleaned all notices,notice,preprocess_boamp_m0.py,84623,93.0,current,3e69c5176e229fcc31e814baf9450c4bbccb22ad9b3e62...
4,m0_sources,data/processed/boamp_m0_sources.csv,M0 digital APPEL_OFFRE sources,source notice,preprocess_boamp_m0.py,3159,22.0,current,9fc77d7ca6da0fb3914a90284ca3da0b3eeb7bcc238e90...
5,candidate_pairs,data/processed/boamp_m0_candidate_pairs.csv,blocked/scored pairs,source-candidate pair,build_m0_candidate_pairs.py,6137,19.0,current,3ce33cb13531046f56ee96e472f5cac1222e923357245d...
6,links_broad,data/processed/boamp_m0_links_broad.csv,broad links,accepted link,run_m0_linkage.py,927,21.0,current,eefb87ea5dc70b5f5b9251c7c3717998d0cff4ad1dfb67...
7,links_balanced,data/processed/boamp_m0_links_balanced.csv,reference balanced links,accepted link,run_m0_linkage.py,618,21.0,current,59fd8434baed5aab6963c8c793c289e6b045514f3b740e...
8,links_strict,data/processed/boamp_m0_links_strict.csv,strict links,accepted link,run_m0_linkage.py,309,21.0,current,127addcae15ec9b98837a5276edac060fdba626a75175f...
9,survival_balanced,data/processed/boamp_survival_m0_balanced.csv,reference survival handoff,source notice,run_m0_linkage.py,3159,18.0,current,93c05a0c767b7264f2ed7669c7b70775f1ec86b7eeddbc...


### `audit_blocking_coverage.csv`

Blocking coverage overall and by year, CPV division, buyer-key type, duration status, and zero-candidate cause.

In [5]:
path = TABLES_DIR / 'audit_blocking_coverage.csv'
df = pd.read_csv(path)
print(df.shape)
display(df.head(20))

(59, 6)


,breakdown,level,n_eligible_sources,n_sources_with_candidate,n_sources_zero_candidate,blocking_coverage
0,overall,eligible_nonmissing_buyer_key,3159,1236,1923,0.391263
1,publication_year,2015,266,151,115,0.567669
2,publication_year,2016,308,177,131,0.574675
3,publication_year,2017,322,191,131,0.593168
4,publication_year,2018,307,156,151,0.508143
5,publication_year,2019,314,165,149,0.525478
6,publication_year,2020,259,109,150,0.420849
7,publication_year,2021,306,95,211,0.310458
8,publication_year,2022,305,60,245,0.196721
9,publication_year,2023,283,18,265,0.063604


### `audit_candidate_source_diagnostics.csv`

One row per eligible source with candidate counts, best score, runner-up score, margin, and zero-candidate cause.

In [6]:
path = TABLES_DIR / 'audit_candidate_source_diagnostics.csv'
df = pd.read_csv(path)
print(df.shape)
display(df.head(20))

(3159, 17)


,source_notice_id,publication_year,cpv_division,buyer_key,buyer_key_type,duration_status,buyer_publication_frequency,n_later_same_buyer_sources,n_later_same_buyer_in_temporal_window,n_scored_candidates,best_score,second_best_score,score_margin,source_has_only_one_candidate,selected_candidate_notice_id,selected_candidate_multiplicity,zero_candidate_cause
0,23-140699,2023,50.0,NAME:3rd anjou,NAME_FALLBACK,imputed,1,0,0,0,NaN,NaN,NaN,False,NaN,0,no_later_same_buyer_notice
1,15-31797,2015,72.0,NAME:acoss,NAME_FALLBACK,imputed,3,2,0,0,NaN,NaN,NaN,False,NaN,0,temporal_window_excluded_all_later_same_buyer
2,22-105844,2022,77.0,NAME:acoss,NAME_FALLBACK,imputed,3,1,0,0,NaN,NaN,NaN,False,NaN,0,temporal_window_excluded_all_later_same_buyer
3,23-167185,2023,50.0,NAME:acoss,NAME_FALLBACK,imputed,3,0,0,0,NaN,NaN,NaN,False,NaN,0,no_later_same_buyer_notice
4,25-86689,2025,48.0,NAME:acquisition maintenance et hebergement d ...,NAME_FALLBACK,observed,1,0,0,0,NaN,NaN,NaN,False,NaN,0,no_later_same_buyer_notice
5,15-69493,2015,72.0,NAME:ademe,NAME_FALLBACK,imputed,10,9,0,0,NaN,NaN,NaN,False,NaN,0,temporal_window_excluded_all_later_same_buyer
6,16-1046,2016,72.0,NAME:ademe,NAME_FALLBACK,imputed,10,8,0,0,NaN,NaN,NaN,False,NaN,0,temporal_window_excluded_all_later_same_buyer
7,16-67267,2016,48.0,NAME:ademe,NAME_FALLBACK,imputed,10,7,0,0,NaN,NaN,NaN,False,NaN,0,temporal_window_excluded_all_later_same_buyer
8,16-93714,2016,30.0,NAME:ademe,NAME_FALLBACK,imputed,10,6,0,0,NaN,NaN,NaN,False,NaN,0,temporal_window_excluded_all_later_same_buyer
9,17-59442,2017,72.0,NAME:ademe,NAME_FALLBACK,imputed,10,5,0,0,NaN,NaN,NaN,False,NaN,0,temporal_window_excluded_all_later_same_buyer


### `audit_duplicate_candidate_summary.csv`

Balanced-link candidate reuse summary.

In [7]:
path = TABLES_DIR / 'audit_duplicate_candidate_summary.csv'
df = pd.read_csv(path)
print(df.shape)
display(df.head(20))

(1, 6)


,variant,n_selected_links,n_unique_selected_candidates,n_candidates_reused,share_candidates_reused,max_multiplicity
0,balanced,618,443,121,0.273138,6


### `audit_survival_construction_checks.csv`

Internal consistency checks for the balanced survival handoff.

In [8]:
path = TABLES_DIR / 'audit_survival_construction_checks.csv'
df = pd.read_csv(path)
print(df.shape)
display(df.head(20))

(12, 3)


,check,value,note
0,survival_rows,3159.0,rows in balanced survival handoff
1,duplicated_survival_notice_ids,0.0,must be zero
2,events,618.0,event rows
3,balanced_links,618.0,accepted links
4,events_equal_links,1.0,1 pass
5,event_rows_missing_candidate,0.0,must be zero
6,censored_rows_with_candidate,0.0,must be zero
7,nonpositive_time,0.0,must be zero
8,event_ids_not_in_links,0.0,must be zero
9,links_not_in_survival,0.0,must be zero


### `audit_censoring_diagnostics.csv`

Observed differences between event and censored groups.

In [9]:
path = TABLES_DIR / 'audit_censoring_diagnostics.csv'
df = pd.read_csv(path)
print(df.shape)
display(df.head(20))

(54, 8)


,variable,type,level,event_mean,censored_mean,standardized_mean_difference,event_share,censored_share
0,declared_duration_months,numeric,NaN,28.487055,33.965368,-0.296133,NaN,NaN
1,n_scored_candidates,numeric,NaN,6.825243,0.755214,1.300851,NaN,NaN
2,buyer_publication_frequency,numeric,NaN,63.352751,19.255805,0.881313,NaN,NaN
3,publication_year,categorical_share,2015,NaN,NaN,NaN,0.132686,0.072412
4,publication_year,categorical_share,2016,NaN,NaN,NaN,0.132686,0.088941
5,publication_year,categorical_share,2017,NaN,NaN,NaN,0.173139,0.084612
6,publication_year,categorical_share,2018,NaN,NaN,NaN,0.134304,0.088154
7,publication_year,categorical_share,2019,NaN,NaN,NaN,0.110032,0.096812
8,publication_year,categorical_share,2020,NaN,NaN,NaN,0.085761,0.081070
9,publication_year,categorical_share,2021,NaN,NaN,NaN,0.066343,0.104290
